## Few Pytorch Operations on arrays 

In [295]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

In [2]:
torch.tensor(3)

tensor(3)

In [3]:
torch.randn(3)

tensor([-1.3887,  0.3374,  0.6374])

In [4]:
torch.randn((4,3))

tensor([[ 0.2828, -0.1807,  0.0622],
        [-0.6159,  1.0154,  0.1127],
        [ 0.4214, -0.2203, -0.6931],
        [-0.4622,  0.1993,  0.2390]])

In [78]:
w = torch.randn(4)
x = torch.tensor([3.0,2.0,3.5,1.0])
torch.dot(w,x)

tensor(5.6953)

In [79]:
torch.randn(4,8).shape[1] 

8

In [80]:
x = torch.tensor([[2.0,3.0],[1.2,5.8],[4.1,6.9]])
y = torch.tensor([1.0,0.0,1.0])
for x_in,y_in in zip(x,y):
    print(x_in,y_in)

tensor([2., 3.]) tensor(1.)
tensor([1.2000, 5.8000]) tensor(0.)
tensor([4.1000, 6.9000]) tensor(1.)


## Creating Perceptron Model From scratch 

In [ ]:
import torch
class Perceptron:
    def __init__(self,X,y,lr,epochs,batch_size):
        self.n_samples,self.n_features = X.shape
        
        self.weights = torch.randn(self.n_features+1)
        self.lr = lr 
        self.epochs = epochs
        self.X = X 
        self.y = y
        self.batch_size = batch_size 

    def step_func(self,z):
        return 1 if z>0 else 0

    def augment(self,x):
        return torch.cat([torch.tensor([1.]),x])
        
    def predict(self,x):
        x = self.augment(x) 
        z = torch.dot(self.weights,x)
        return self.step_func(z) 

    def fit(self):
        epoch_counter = 1 
        for _ in range(1,self.epochs+1):
            epoch_loss = 0 

            batch_counter = 1
            for batch_start in range(0,self.n_samples,self.batch_size):
                
                batch_end = batch_start + self.batch_size 
                
                x_batch  = self.X[batch_start:batch_end] 
                y_batch  = self.y[batch_start:batch_end] 
                
                # compute the prediction
                for x,y in zip(x_batch,y_batch):
                    
                    prediction = self.predict(x)
                    
                    error = y-prediction
                    
                    epoch_loss += abs(error)
                    
                    # update the weights
                    x = self.augment(x) 
                    self.weights = self.weights + (self.lr * error * x)
                    
                # print(f'\tBatch = {batch_counter} - completed')
                batch_counter +=1 
                          
            print(f'eopch = {epoch_counter} ; loss = {epoch_loss}')
            epoch_counter +=1

X = torch.tensor([
    [1., 45.],
    [2., 50.],
    [2., 55.],
    [3., 60.],
    [4., 65.],
    [5., 70.],
    [6., 75.],
    [7., 80.],
    [8., 85.],
    [9., 90.]
])

y = torch.tensor([
    0., 0., 0., 0., 0.,
    1., 1., 1., 1., 1.
])

model = Perceptron(X,y,lr=0.001,epochs = 20,batch_size = 2) 
model.fit()

# prediction for a single input 
test_prediction = model.predict(torch.tensor([2.,48.]))
print('test_prediction = ',test_prediction) 

print('testing multiple samples') 
print('*'*50) 
test_samples = torch.tensor([
    [2., 48.],   # Expected Fail
    [3., 72.],   # Borderline
    [6., 78.],   # Expected Pass
    [8., 88.]    # Expected Pass
])

for sample in test_samples:
    print(sample.tolist(), "->", model.predict(sample))

### Creating a small Neural Network using PyTorch

In [107]:
import torch

X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0],
    [6.0],
    [7.0],
    [8.0]
])

y = torch.tensor([
    [0.0],
    [0.0],
    [0.0],
    [0.0],
    [1.0],
    [1.0],
    [1.0],
    [1.0]
])

In [266]:
torch.manual_seed(101) 
random.seed(101)
class SingleNeuron(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.linear1 = torch.nn.Linear(1, 2,bias=True)
        self.linear2 = torch.nn.Linear(2,1,bias = False)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):

        a = self.linear1(x)
        b = self.linear2(a)
        y_hat = self.sigmoid(b)

        return y_hat

model = SingleNeuron()

In [267]:
x = torch.tensor([3.8])
model(x)

tensor([0.6586], grad_fn=<SigmoidBackward0>)

In [268]:
model.linear1.weight

Parameter containing:
tensor([[-0.6039],
        [-0.0994]], requires_grad=True)

## Adding criterion, optimizer, training loop

In [269]:
torch.manual_seed(101)
random.seed(101) 

criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)
epochs = 5

for epoch in range(epochs):

    predictions = model(X)  # find predictions 
    loss = criterion(predictions, y) # find loss 
    optimizer.zero_grad() # apply zero grad 
    loss.backward() # loss backward 
    optimizer.step() # optimizer step 

    print(f"Epoch {epoch+1:3d} ; Loss = {loss.item():.4f}") 

# Epoch   1 ; Loss = 0.5630
# Epoch   2 ; Loss = 0.5558
# Epoch   3 ; Loss = 0.5514
# Epoch   4 ; Loss = 0.5476
# Epoch   5 ; Loss = 0.5439

Epoch   1 ; Loss = 0.5630
Epoch   2 ; Loss = 0.5558
Epoch   3 ; Loss = 0.5514
Epoch   4 ; Loss = 0.5476
Epoch   5 ; Loss = 0.5439


## Understanding linear layer 

 * Applies an affine linear transformation to the incoming data: $xA^T+b$
 * Values are initialized using $\mathbb{U}\left(-\sqrt{k},+\sqrt{k}\right)$ , where $k$ is the number of input features.

In [270]:
torch.manual_seed(101)
random.seed(101)

print("Weight:", model.linear1.weight)
print("Bias:", model.linear1.bias)

Weight: Parameter containing:
tensor([[-0.6076],
        [-0.1113]], requires_grad=True)
Bias: Parameter containing:
tensor([-0.7970,  0.8297], requires_grad=True)


In [271]:
model.linear1.reset_parameters() # goes back to the initialization step 

In [272]:
model.linear1.weight

Parameter containing:
tensor([[-0.6039],
        [-0.0994]], requires_grad=True)

In [258]:
model.linear1.weight.shape

torch.Size([2, 1])

In [259]:
model.linear1.bias.shape

torch.Size([2])

In [276]:
import random 
random.seed(101)
torch.manual_seed(101)

LinearLayer = torch.nn.Linear(1,2)
LinearLayer.weight

Parameter containing:
tensor([[-0.6039],
        [-0.0994]], requires_grad=True)

In [277]:
x = torch.tensor([4.0])

In [282]:
LinearLayer(x)

tensor([-3.2338,  0.3767], grad_fn=<ViewBackward0>)

In [281]:
x@LinearLayer.weight.T.detach() + LinearLayer.bias.detach()

tensor([-3.2338,  0.3767])

In [283]:
model.linear1.extra_repr()

'in_features=1, out_features=2, bias=True'

In [274]:
test = torch.tensor([
    [2.5],
    [4.5],
    [6.5]
])

with torch.no_grad():
    probabilities = model(test)
    predictions = (probabilities >= 0.5).float()

for x, p, y_hat in zip(test, probabilities, predictions):
    print(f"Hours: {x.item():.1f} | Probability: {p.item():.3f} | Prediction: {int(y_hat.item())}")

Hours: 2.5 | Probability: 0.544 | Prediction: 1
Hours: 4.5 | Probability: 0.650 | Prediction: 1
Hours: 6.5 | Probability: 0.743 | Prediction: 1


## Multi - Layer - Perceptron 

In [14]:
import torch
import torch.nn as nn
class StudentMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 4),
            nn.ReLU(),
            nn.Linear(4, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

In [15]:
X = torch.tensor([

    [2.,40.,1.],
    [3.,50.,2.],
    [3.,60.,3.],
    [4.,65.,4.],
    [5.,70.,5.],
    [6.,75.,6.],
    [7.,80.,7.],
    [8.,90.,8.]

])

y = torch.tensor([

    [0.],
    [0.],
    [0.],
    [0.],
    [1.],
    [1.],
    [1.],
    [1.]

])

In [16]:
model = StudentMLP()
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [17]:
epochs = 500

for epoch in range(epochs):

    predictions = model(X)

    loss = criterion(predictions, y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch+1)%50==0:

        print(
            epoch+1,
            loss.item()
        )

50 0.46020716428756714
100 0.19866807758808136
150 0.11267899721860886
200 0.07100749015808105
250 0.05099349468946457
300 0.04012157768011093
350 0.030536357313394547
400 0.024780914187431335
450 0.020643362775444984
500 0.017295731231570244


In [18]:
test = torch.tensor([
    [2.,45.,2.],
    [4.,68.,5.],
    [6.,78.,6.],
    [8.,92.,8.]
])

with torch.no_grad():
    probability = model(test)
    prediction = (probability >=0.5).float()

for x,pred,prob in zip(test,prediction,probability):
    print(x.tolist(), pred.item(), prob.item())

[2.0, 45.0, 2.0] 0.0 0.032368578016757965
[4.0, 68.0, 5.0] 1.0 0.5618336796760559
[6.0, 78.0, 6.0] 1.0 0.9999661445617676
[8.0, 92.0, 8.0] 1.0 0.9999995231628418
